In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import pandas as pd
import blimpy as bl
from blimpy import Waterfall
from pathlib import Path
import os

# Mars Inventory Csv:

In [ ]:
midres_dir = Path("/datag/blpd18/datax/gb_mars/splices")

rows = []

for f in sorted(midres_dir.glob("*.0002.fil")):
    wf = Waterfall(str(f), load_data=False)
    h = wf.header
    
    rows.append({
        "file": f.name,
        "source": h.get("source_name"),
        "nchans": h["nchans"],
        "foff_Hz": h["foff"] * 1e6,  # MHz -> Hz
        "tsamp_s": h["tsamp"],
        "fch1_MHz": h["fch1"],
        "bandwidth_MHz": abs(h["foff"]) * h["nchans"]
    })

df = pd.DataFrame(rows)

df

In [ ]:
narrow_dir = Path("/datag/blpd18/datax/gb_mars/single_coarse_channel")

rows = []

for f in sorted(narrow_dir.glob("*.h5")):
    wf = Waterfall(str(f), load_data=False)
    h = wf.header
    
    rows.append({
        "file": f.name,
        "source": h.get("source_name"),
        "nchans": h["nchans"],
        "foff_Hz": h["foff"] * 1e6,
        "tsamp_s": h["tsamp"],
        "fch1_MHz": h["fch1"],
        "bandwidth_MHz": abs(h["foff"]) * h["nchans"]
    })

narrow_df = pd.DataFrame(rows)

narrow_df

In [ ]:

for f in sorted(Path("/datag/blpd18/datax/gb_mars/splices").glob("*.0002.fil")):
    wf = Waterfall(str(f), load_data=False)
    h = wf.header

    print(f.name)
    print("  source :", h.get("source_name"))
    print("  nchans :", h["nchans"])
    print("  foff   :", h["foff"])
    print("  tsamp  :", h["tsamp"])
    print("  fch1   :", h["fch1"])
    print()

# Quick Look Plots:

In [ ]:
mid_file = "/datag/blpd18/datax/gb_mars/splices/spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66265_DIAG_Mars2020_0011.rawspec.0002.fil"

narrow_file = "/datag/blpd18/datax/gb_mars/single_coarse_channel/spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66265_DIAG_Mars2020_0011.rawspec.0000_diced.h5"

In [ ]:
mid = Waterfall(mid_file, load_data=False)
nar = Waterfall(narrow_file, load_data=False)

print(mid.header)
print(nar.header)

In [ ]:
mid_data = Waterfall(mid_file).data
nar_data = Waterfall(narrow_file).data

print(mid_data.shape)
print(nar_data.shape)

In [ ]:
mid_freq = (
    mid.header["fch1"]
    + np.arange(mid.header["nchans"]) * mid.header["foff"]
)

nar_freq = (
    nar.header["fch1"]
    + np.arange(nar.header["nchans"]) * nar.header["foff"]
)

print(mid_freq[0], mid_freq[-1])
print(nar_freq[0], nar_freq[-1])

In [ ]:
from turbo_seti.find_event.find_event_pipeline import find_event_pipeline

events = find_event_pipeline(
    "mars_single_coarse.lst",
    filter_threshold=3,
    check_zero_drift=False,
    SNR_cut=10,
    on_off_first="ON",
    number_in_cadence=6,
    saving=True
)

print(events)

In [ ]:
#!/usr/bin/env python3

import pandas as pd
import os

# ---------------------------------------------------------
# Path to BLISS output CSV
# ---------------------------------------------------------

csv_path = (
    "/home/wlll2x/midres_narrow/mars_single_coarse_bliss/results_mars_single_coarse/"
    "DIAG_Mars2020_f1_snr3_zero.csv"
)

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"Could not find {csv_path}\n"
        "Run: find ~/midres_narrow -name 'DIAG_Mars2020_f1_snr3_zero.csv'"
    )


# ---------------------------------------------------------
# Load BLISS hits
# ---------------------------------------------------------

df = pd.read_csv(csv_path)

print("\nLoaded BLISS hits:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())


# ---------------------------------------------------------
# Known Mars signal recovery test
#
# This is NOT a blind search.
# It checks whether BLISS recovered the
# known turboSETI candidate.
# ---------------------------------------------------------

mars_candidate = df[
    (df["Freq"] > 8414.40) &
    (df["Freq"] < 8414.55) &
    (df["DriftRate"] > -1.0) &
    (df["DriftRate"] < 1.0)
]


print("\n====================================")
print("Known Mars frequency region")
print("====================================")

if len(mars_candidate) == 0:
    print("No hits found near known Mars candidate.")
else:
    print(
        mars_candidate[
            [
                "DriftRate",
                "SNR",
                "Freq",
                "FileID",
                "status"
            ]
        ]
        .sort_values(
            "SNR",
            ascending=False
        )
        .head(30)
        .to_string(index=False)
    )


# ---------------------------------------------------------
# Compare ON/OFF behavior
# ---------------------------------------------------------

print("\n====================================")
print("File distribution")
print("====================================")

if len(mars_candidate) > 0:
    print(
        mars_candidate.groupby("FileID").size()
    )


# ---------------------------------------------------------
# Highest SNR hits overall
# ---------------------------------------------------------

print("\n====================================")
print("Highest SNR BLISS hits")
print("====================================")

print(
    df[
        [
            "DriftRate",
            "SNR",
            "Freq",
            "FileID"
        ]
    ]
    .sort_values(
        "SNR",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)


print("\nDone.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


csv_path = (
    "/home/wlll2x/midres_narrow/mars_single_coarse_bliss/results_mars_single_coarse/"
    "DIAG_Mars2020_f1_snr3_zero.csv"
)

df = pd.read_csv(csv_path)


# Select recovered Mars candidate
mars = df[
    (df["Freq"] > 8414.46) &
    (df["Freq"] < 8414.48) &
    (df["DriftRate"] > -0.4) &
    (df["DriftRate"] < -0.2)
]


print(mars[
    [
        "Freq",
        "DriftRate",
        "SNR",
        "status"
    ]
])


plt.figure(figsize=(8,5))


for label, group in mars.groupby("status"):
    plt.scatter(
        group["Freq"],
        group["DriftRate"],
        s=80,
        label=label
    )


plt.axhline(
    -0.309,
    linestyle="--",
    label="TurboSETI drift"
)


plt.xlabel("Frequency (MHz)")
plt.ylabel("Drift Rate (Hz/s)")
plt.title("BLISS Recovery of Mars 2020 Signal")

plt.legend()
plt.grid()

plt.savefig(
    "BLISS_Mars_drift_recovery.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
mars = df[
    (df["Freq"].between(8414.46, 8414.48)) &
    (df["DriftRate"].between(-0.4, -0.2)) &
    (df["status"].str.contains("table"))
]

print(len(mars))

display(
    mars[
        [
            "Freq",
            "DriftRate",
            "SNR",
            "status"
        ]
    ].sort_values("SNR", ascending=False)
)

In [1]:
import pandas as pd
import os

from turbo_seti.find_event.plot_event_pipeline import plot_event_pipeline


# ============================
# Load BLISS results
# ============================

csv_path = (
    "/home/wlll2x/midres_narrow/mars_single_coarse_bliss/results_mars_single_coarse/"
    "DIAG_Mars2020_f1_snr3_zero.csv"
)

df = pd.read_csv(csv_path)

print(f"Loaded {len(df)} BLISS hits")



# ============================
# Mars2020 expected parameters
# ============================

expected_freq = 8414.4689       # MHz
expected_drift = -0.309         # Hz/s


freq_window = 0.01              # MHz
drift_window = 0.05              # Hz/s



# ============================
# Find Mars-like candidates
# ============================

mars_candidates = df[
    (abs(df["Freq"] - expected_freq) < freq_window) &
    (abs(df["DriftRate"] - expected_drift) < drift_window)
].copy()


print(
    f"Found {len(mars_candidates)} Mars-like candidates"
)


print(
    mars_candidates[
        ["Freq", "DriftRate", "SNR", "status"]
    ]
    .sort_values(
        "SNR",
        ascending=False
    )
    .head(20)
)



# ============================
# Pick strongest Mars candidate
# ============================

if len(mars_candidates) == 0:
    raise RuntimeError(
        "No Mars-like candidates found. Check frequency/drift windows."
    )


best = mars_candidates.sort_values(
    "SNR",
    ascending=False
).iloc[0]


print("\nBest Mars candidate:")
print(best)



# ============================
# Save candidate
# ============================

candidate_csv = (
    "/home/wlll2x/mars_best_candidate.csv"
)


best.to_frame().T.to_csv(
    candidate_csv,
    index=False
)



# ============================
# Plot
# ============================

plot_dir = (
    "/home/wlll2x/mars_best_candidate_plot"
)


os.makedirs(
    plot_dir,
    exist_ok=True
)


plot_event_pipeline(
    candidate_csv,
    "/home/wlll2x/midres_narrow/mars_single_coarse_correct.lst",
    filter_spec="f3",
    user_validation=False,
    plot_dir=plot_dir
)


print("Done!")

/home/wlll2x/.conda/envs/willvenv/lib/python3.10/site-packages/blimpy/__init__.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


FileNotFoundError: [Errno 2] No such file or directory: '/home/wlll2x/midres_narrow/mars_single_coarse_bliss/results_mars_single_coarse/DIAG_Mars2020_f1_snr3_zero.csv'

In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt

target_freq = 8414.46861  # MHz
window = 5.0              # MHz

files = glob.glob(
    "/datax/scratch/wlll2x/mars_midres_bliss/*.dat"
)

columns = [
    "hit",
    "drift_rate",
    "snr",
    "freq",
    "corrected_freq",
    "index",
    "freq_start",
    "freq_end",
    "sefd_freq",
    "coarse_channel",
    "full_number_hits",
    "extra"
]

hits = []

for f in files:
    df = pd.read_csv(
        f,
        comment="#",
        sep=r"\s+",
        header=None
    )

    df.columns = columns
    df["file"] = f
    hits.append(df)

df = pd.concat(hits, ignore_index=True)


# Select only hits near Mars frequency
near_mars = df[
    abs(df["freq"] - target_freq) < window
]


print(f"Total BLISS hits: {len(df)}")
print(f"Hits within ±{window} MHz of Mars: {len(near_mars)}")

print(
    near_mars[
        [
            "freq",
            "drift_rate",
            "snr",
            "full_number_hits",
            "file"
        ]
    ].sort_values("snr", ascending=False)
)


# Plot if anything exists
if len(near_mars) > 0:

    plt.figure(figsize=(10,6))

    plt.scatter(
        near_mars["freq"],
        near_mars["snr"],
        c=near_mars["drift_rate"],
        s=80
    )

    plt.axvline(
        target_freq,
        linestyle="--",
        label="Expected Mars frequency"
    )

    plt.xlabel("Frequency (MHz)")
    plt.ylabel("SNR")
    plt.colorbar(label="Drift rate (Hz/s)")
    plt.legend()
    plt.title(
        f"BLISS hits within ±{window} MHz of Mars frequency"
    )

    plt.show()

else:
    print("No BLISS hits found")

# MidRes Plotter

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/datax/scratch/wlll2x/mars_midres_bliss/spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66265_DIAG_Mars2020_0011.rawspec.0002.dat",
    comment="#",
    sep=r"\s+",
    header=None
)

df.columns = [
    "Top_Hit_#",          # hit number
    "Drift_Rate",
    "SNR",
    "Freq",
    "Freq_start",
    "Index",
    "freq_start",
    "freq_end",
    "SEFD",
    "Coarse",
    "NumHits",
    "Extra"
]

print(df.head())

# here is the coarse channel plotter for the correct freq

In [1]:
from blimpy import Waterfall
import matplotlib.pyplot as plt
import glob
import os
import numpy as np

target_freq = 8414.468571  # MHz
window = 0.01              # MHz = 10 kHz

files = sorted(glob.glob(
    "/datax/scratch/wlll2x/mars_midres/*.h5"
))

for filename in files:

    print("\nProcessing:", os.path.basename(filename))

    wf = Waterfall(filename)

    fch1 = wf.header["fch1"]
    foff = wf.header["foff"]

    print("fch1:", fch1)
    print("foff:", foff)

    # Crop around target
    f_start = target_freq - window
    f_stop  = target_freq + window

    try:
        wf = Waterfall(
            filename,
            f_start=f_start,
            f_stop=f_stop
        )

    except Exception as e:
        print("Crop failed:", e)
        continue


    data = wf.data.squeeze()

    freqs = (
        wf.header["fch1"]
        + np.arange(data.shape[-1]) * wf.header["foff"]
    )

    plt.figure(figsize=(10,5))

    plt.imshow(
        10*np.log10(data),
        aspect="auto",
        extent=[
            freqs[0],
            freqs[-1],
            0,
            data.shape[0]
        ],
        origin="lower"
    )

    plt.axvline(
        target_freq,
        linestyle="--",
        label="8414.468571 MHz"
    )

    plt.xlabel("Frequency (MHz)")
    plt.ylabel("Time index")
    plt.title(os.path.basename(filename))

    plt.legend()
    plt.colorbar(label="Power (dB)")

    plt.tight_layout()

    out = (
        os.path.basename(filename)
        .replace(".h5",".png")
    )

    plt.savefig(out,dpi=200)
    plt.close()

    print("Saved:",out)

/home/wlll2x/.conda/envs/willvenv/lib/python3.10/site-packages/blimpy/__init__.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound



Processing: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66265_DIAG_Mars2020_0011.rawspec.0002.h5
fch1: 11251.46484375
foff: -0.00286102294921875
Saved: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66265_DIAG_Mars2020_0011.rawspec.0002.png

Processing: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66578_DIAG_Mars_0012.rawspec.0002.h5
fch1: 11251.46484375
foff: -0.00286102294921875
Saved: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66578_DIAG_Mars_0012.rawspec.0002.png

Processing: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66892_DIAG_Mars2020_0013.rawspec.0002.h5
fch1: 11251.46484375
foff: -0.00286102294921875
Saved: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66892_DIAG_Mars2020_0013.rawspec.0002.png

Processing: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_67205_DIAG_Mars_0014.raw

In [2]:
fch1 = 11251.46484375
foff = -0.00286102294921875

target = 8414.468571

chan = int((target-fch1)/foff)

print(chan)
print(chan//1024)
print(chan%1024)

991602
968
370


In [ ]:
candidate = df.iloc[[100]]

candidate.to_csv(
    "/home/wlll2x/midres_candidate.csv",
    index=False
)

candidate

In [3]:
from turbo_seti.find_event.find_event_pipeline import find_event_pipeline

events = find_event_pipeline(
    dat_file_list_str="../data/mars_midres_coarse968_dat.lst",
    h5_file_list_str="../data/mars_midres_coarse968_h5.lst",

    number_in_cadence=6,
    on_off_first="ON",

    check_zero_drift=True,
    filter_threshold=1,

    saving=True,
    csv_name="../outputs/midres_coarse968_events.csv"
)

print(events)


===========   BEGINNING FIND_EVENT PIPELINE   ===========

Assuming the first observation is an ON
find_event_pipeline INFO     file=h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66265_DIAG_Mars2020_0011.rawspec.0002_coarse968.dat, tstart=59256.76695601852, source_name=DIAG_Mars2020, fch1=8415.52734375, foff=-0.00286102294921875, nchans=1024
find_event_pipeline INFO     file=h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66578_DIAG_Mars_0012.rawspec.0002_coarse968.dat, tstart=59256.770578703705, source_name=DIAG_Mars, fch1=8415.52734375, foff=-0.00286102294921875, nchans=1024
find_event_pipeline INFO     file=h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66892_DIAG_Mars2020_0013.rawspec.0002_coarse968.dat, tstart=59256.77421296296, source_name=DIAG_Mars2020, fch1=8415.52734375, foff=-0.00286102294921875, nchans=1024
find_event_pipeline INFO     file=h5spliced_blc10111213141516o7o0212223242526o7o03132333

In [7]:
import os
import pandas as pd
from turbo_seti.find_event.plot_event_pipeline import plot_event_pipeline


# ============================
# Load events CSV
# ============================

events = pd.read_csv("../outputs/midres_coarse968_events.csv")


# ============================
# Fix file paths
# ============================

base_path = "/datax/scratch/wlll2x/mars_midres_coarse968"

events["FileID"] = events["FileID"].apply(
    lambda x: os.path.join(
        base_path,
        os.path.basename(x)
    )
)


# ============================
# Check files exist
# ============================

missing = events[~events["FileID"].apply(os.path.exists)]

if len(missing) > 0:
    print("WARNING: Missing files:")
    print(missing["FileID"].unique())
else:
    print("All H5 files found!")


# ============================
# Remove duplicate detections
# Keep strongest SNR per frequency/file
# ============================

events["freq_group"] = events["Freq"].round(3)

unique_events = (
    events
    .sort_values("SNR", ascending=False)
    .drop_duplicates(
        subset=["FileID", "freq_group"]
    )
)


print(
    f"\nOriginal events: {len(events)}"
)

print(
    f"Unique events: {len(unique_events)}"
)


# ============================
# Select strongest events
# ============================

top_events = (
    unique_events
    .sort_values("SNR", ascending=False)
    .head(20)
)


print("\nTop events:")
print(
    top_events[
        [
            "Freq",
            "DriftRate",
            "SNR"
        ]
    ]
)


# ============================
# Plot events
# ============================

for i, row in top_events.iterrows():

    print(
        f"\nPlotting event {i}"
    )

    print(
        f"Freq: {row['Freq']:.6f} MHz"
    )

    print(
        f"Drift: {row['DriftRate']:.3f} Hz/s"
    )

    print(
        f"SNR: {row['SNR']:.2f}"
    )


    plot_event_pipeline(
        row["FileID"],
        row["Freq"],
        row["DriftRate"],
        row["SNR"]
    )

All H5 files found!

Original events: 88
Unique events: 47

Top events:
           Freq  DriftRate         SNR
79  8414.491664  19.169318  702.159668
63  8414.435848 -28.753978  695.375916
66  8414.441601  28.753978  629.954346
61  8414.481624 -28.753978  600.084167
37  8414.435848 -28.753978  555.738953
73  8414.471647  38.338637  546.169434
29  8414.490238  28.753978  545.481445
83  8414.434412 -38.338637  544.568420
35  8414.481624 -28.753978  489.970337
31  8414.441601  28.753978  488.207489
80  8414.490197 -47.923296  468.973602
84  8414.470221  47.923296  450.013794
81  8414.432977 -47.923296  446.800232
42  8414.437273 -38.338637  435.428925
45  8414.471647  38.338637  428.237488
40  8414.488772 -38.338637  375.786438
54  8414.470221  47.923296  351.694794
26  8414.440145 -19.169318  336.235657
11  8414.490238  28.753978  322.053406
17  8414.481624 -28.753978  297.978729

Plotting event 79
Freq: 8414.491664 MHz
Drift: 19.169 Hz/s
SNR: 702.16
*** plot_event_pipeline: Oops, cannot

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from blimpy import Waterfall

# ==========================================================
# Inputs
# ==========================================================

event_csv = "../outputs/midres_coarse968_events.csv"

files = [
"/datax/scratch/wlll2x/mars_midres_coarse968/h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66265_DIAG_Mars2020_0011.rawspec.0002_coarse968.h5",
"/datax/scratch/wlll2x/mars_midres_coarse968/h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66578_DIAG_Mars_0012.rawspec.0002_coarse968.h5",
"/datax/scratch/wlll2x/mars_midres_coarse968/h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66892_DIAG_Mars2020_0013.rawspec.0002_coarse968.h5",
"/datax/scratch/wlll2x/mars_midres_coarse968/h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_67205_DIAG_Mars_0014.rawspec.0002_coarse968.h5",
"/datax/scratch/wlll2x/mars_midres_coarse968/h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_67519_DIAG_Mars2020_0015.rawspec.0002_coarse968.h5",
"/datax/scratch/wlll2x/mars_midres_coarse968/h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_67832_DIAG_Mars_0016.rawspec.0002_coarse968.h5",
]

target_freq = 8414.468992
zoom_MHz = 0.1          # +/-5 kHz

# ==========================================================
# BLISS hits
# ==========================================================

events = pd.read_csv(event_csv)

events["distance"] = np.abs(events["Freq"] - target_freq)

closest = (
    events
    .sort_values("distance")
    .head(5)
)

print(closest[["Freq","DriftRate","SNR"]])

# ==========================================================
# Figure
# ==========================================================

fig, axes = plt.subplots(
    len(files),
    1,
    figsize=(14,18),
    sharex=True,
    gridspec_kw={"hspace":0.04},
)

if len(files) == 1:
    axes = [axes]

# ==========================================================
# Determine ONE common color scale
# ==========================================================

vmins = []
vmaxs = []

for filename in files:

    wf = Waterfall(filename)

    data = wf.data[:,0,:]

    fch1 = wf.header["fch1"]
    foff = wf.header["foff"]

    freqs = fch1 + np.arange(data.shape[1])*foff

    mask = (
        (freqs > target_freq-zoom_MHz) &
        (freqs < target_freq+zoom_MHz)
    )

    crop = data[:,mask]

    db = 10*np.log10(crop)

    vmins.append(np.nanpercentile(db,5))
    vmaxs.append(np.nanpercentile(db,99.5))

vmin = min(vmins)
vmax = max(vmaxs)

# ==========================================================
# Plot each cadence file
# ==========================================================

for ax, filename in zip(axes, files):

    print(Path(filename).name)

    wf = Waterfall(filename)

    data = wf.data[:,0,:]

    times = np.arange(data.shape[0])*wf.header["tsamp"]

    fch1 = wf.header["fch1"]
    foff = wf.header["foff"]

    freqs = fch1 + np.arange(data.shape[1])*foff

    mask = (
        (freqs > target_freq-zoom_MHz) &
        (freqs < target_freq+zoom_MHz)
    )

    crop = data[:,mask]
    crop_freqs = freqs[mask]

    if crop.shape[1] == 0:
        print("No channels!")
        continue

    image = 10*np.log10(crop[::-1,::-1])

    im = ax.imshow(
        image,
        aspect="auto",
        origin="lower",
        extent=[
            crop_freqs[-1],
            crop_freqs[0],
            times[-1],
            times[0]
        ],
        vmin=vmin,
        vmax=vmax,
    )

    # ------------------------
    # Overlay BLISS tracks
    # ------------------------

    for _, row in closest.iterrows():

        drift = row["DriftRate"]/1e6

        track = row["Freq"] + drift*times

        ax.plot(
            track,
            times,
            linewidth=2,
            alpha=0.9,
        )

        ax.scatter(
            row["Freq"],
            times[0],
            s=35,
        )

    # ------------------------
    # Labels
    # ------------------------

    source = wf.header["source_name"]

    ax.text(
        0.98,
        0.92,
        source,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=13,
        fontweight="bold",
        color="white",
        bbox=dict(
            facecolor="black",
            alpha=0.7,
            boxstyle="round"
        )
    )

    label = "ON" if "Mars2020" in source else "OFF"

    color = "lime" if label=="ON" else "orange"

    ax.text(
        0.02,
        0.92,
        label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=14,
        fontweight="bold",
        color=color,
        bbox=dict(
            facecolor="black",
            alpha=0.7,
            boxstyle="round"
        )
    )

    ax.set_ylabel("Time (s)")

axes[-1].set_xlabel("Frequency (MHz)")

fig.colorbar(
    im,
    ax=axes,
    label="Power (dB)",
    shrink=0.95,
)

plt.suptitle(
    f"Midres cadence centered at {target_freq:.6f} MHz",
    fontsize=16,
)

plt.savefig(
    "../outputs/mars_midres_coarse968_ON_OFF_cadence.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

           Freq  DriftRate         SNR
15  8414.468745 -38.338637   74.842705
41  8414.468745 -38.338637  123.975471
82  8414.468745 -38.338637  155.815170
49  8414.470221  47.923296  168.854645
57  8414.470221  47.923296  309.001984
h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66265_DIAG_Mars2020_0011.rawspec.0002_coarse968.h5
h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66578_DIAG_Mars_0012.rawspec.0002_coarse968.h5
h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66892_DIAG_Mars2020_0013.rawspec.0002_coarse968.h5
h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_67205_DIAG_Mars_0014.rawspec.0002_coarse968.h5
h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_67519_DIAG_Mars2020_0015.rawspec.0002_coarse968.h5
h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_67832_DIAG_Mars_0016.rawspec.0002_coarse968.h5


In [10]:
import pandas as pd

events = pd.read_csv("../outputs/midres_coarse968_events.csv")

base = "/datax/scratch/wlll2x/mars_midres_coarse968"

events["FileID"] = events["FileID"].apply(
    lambda x: f"{base}/{x.split('/')[-1]}"
)

lst_file = "../outputs/midres_coarse968_files.lst"

with open(lst_file, "w") as f:
    for filename in events["FileID"].unique():
        f.write(filename + "\n")

print("Created:", lst_file)

Created: ../outputs/midres_coarse968_files.lst


# ON/OFF Graph

In [1]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from blimpy import Waterfall

# --------------------------------------------------
# Inputs
# --------------------------------------------------

event_csv = "../outputs/midres_coarse968_events.csv"

files = [
"/datax/scratch/wlll2x/mars_midres_coarse968/h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66265_DIAG_Mars2020_0011.rawspec.0002_coarse968.h5",
"/datax/scratch/wlll2x/mars_midres_coarse968/h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66578_DIAG_Mars_0012.rawspec.0002_coarse968.h5",
"/datax/scratch/wlll2x/mars_midres_coarse968/h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66892_DIAG_Mars2020_0013.rawspec.0002_coarse968.h5",
"/datax/scratch/wlll2x/mars_midres_coarse968/h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_67205_DIAG_Mars_0014.rawspec.0002_coarse968.h5",
"/datax/scratch/wlll2x/mars_midres_coarse968/h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_67519_DIAG_Mars2020_0015.rawspec.0002_coarse968.h5",
"/datax/scratch/wlll2x/mars_midres_coarse968/h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_67832_DIAG_Mars_0016.rawspec.0002_coarse968.h5",
]

target_freq = 8414.468992

# 5 kHz window total
span = 0.0025

# --------------------------------------------------
# Find closest BLISS hits
# --------------------------------------------------

events = pd.read_csv(event_csv)

events["distance"] = np.abs(events["Freq"] - target_freq)

closest = (
    events
    .sort_values("distance")
    .head(5)
)

print("\nClosest BLISS hits:\n")
print(closest[["Freq","DriftRate","SNR"]])

# --------------------------------------------------
# Plot
# --------------------------------------------------

fig, axes = plt.subplots(
    len(files),
    1,
    figsize=(15,18),
    sharex=True,
    gridspec_kw={"hspace":0.05}
)

if len(files) == 1:
    axes = [axes]

for ax, filename in zip(axes, files):

    print("Plotting:", Path(filename).name)

    wf = Waterfall(
        filename,
        f_start=target_freq-span,
        f_stop=target_freq+span
    )

    data = wf.data[:,0,:]

    freqs = (
        wf.header["fch1"]
        + np.arange(data.shape[1]) * wf.header["foff"]
    )

    times = (
        np.arange(data.shape[0])
        * wf.header["tsamp"]
    )

    plt.sca(ax)

    ax.imshow(
        10*np.log10(data),
        aspect="auto",
        origin="lower",
        extent=[
            freqs[0],
            freqs[-1],
            times[0],
            times[-1]
        ]
    )

    # Steve convention
    ax.invert_yaxis()
    ax.invert_xaxis()

    # ----------------------------------------
    # Overlay BLISS tracks
    # ----------------------------------------

    for _, row in closest.iterrows():

        freq0 = row["Freq"]

        drift = row["DriftRate"] / 1e6

        freq_track = (
            freq0
            + drift*times
        )

        ax.plot(
            freq_track,
            times,
            linewidth=2
        )

        ax.scatter(
            freq0,
            times[0],
            s=25
        )

    # ----------------------------------------
    # Labels
    # ----------------------------------------

    source = wf.header["source_name"]

    ax.text(
        0.98,
        0.92,
        source,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=14,
        fontweight="bold",
        color="white",
        bbox=dict(
            facecolor="black",
            alpha=0.7
        )
    )

    is_on = "Mars2020" in source

    label = "ON" if is_on else "OFF"

    label_color = (
        "lime"
        if is_on
        else "orange"
    )

    ax.text(
        0.02,
        0.92,
        label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=14,
        fontweight="bold",
        color=label_color,
        bbox=dict(
            facecolor="black",
            alpha=0.7
        )
    )

    ax.set_title("")

    if ax != axes[-1]:
        ax.set_xlabel("")

axes[-1].set_xlabel(
    "Frequency (MHz)",
    fontsize=14
)

plt.savefig(
    "../outputs/mars_coarse968_ON_OFF_stack.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

/home/wlll2x/.conda/envs/willvenv/lib/python3.10/site-packages/blimpy/__init__.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound



Closest BLISS hits:

           Freq  DriftRate         SNR
15  8414.468745 -38.338637   74.842705
41  8414.468745 -38.338637  123.975471
82  8414.468745 -38.338637  155.815170
49  8414.470221  47.923296  168.854645
57  8414.470221  47.923296  309.001984
Plotting: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66265_DIAG_Mars2020_0011.rawspec.0002_coarse968.h5
Plotting: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66578_DIAG_Mars_0012.rawspec.0002_coarse968.h5
Plotting: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_66892_DIAG_Mars2020_0013.rawspec.0002_coarse968.h5
Plotting: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_67205_DIAG_Mars_0014.rawspec.0002_coarse968.h5
Plotting: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi_59256_67519_DIAG_Mars2020_0015.rawspec.0002_coarse968.h5
Plotting: h5spliced_blc10111213141516o7o0212223242526o7o031323334353637_guppi

# Average Spectrum:

# Brightest Channel Track:

# Independent Drift Fitting

# Bliss Comparison: